# 04 · RQ2 - Does the EEG score survive when confounding is removed by design?

The out-of-fold EEG score of notebook 02 is evaluated, without retraining, in subsets or weightings where the
demographics no longer differ between AD and controls (confound-free evaluation; Chyzhyk et al., 2022):

- **RQ2a (BrainLat):** 1:1 education-matched pairs (pair-bootstrap CI; plus 200 random tie-breaks), concordance in
  comparable pairs (education within 2 years), overlap weights for age, sex and education, and the covariate-adjusted
  AUC (check).
- **RQ2b (Greek):** overlap weights for age and sex, comparable pairs (same sex, age within 2 years) and the
  covariate-adjusted AUC (check).

The design (pairs, weights) was fixed in notebook 01, before any prediction; this notebook verifies it.

**Outputs:** outputs/tables/T2b_confounding_by_design.csv.

In [ ]:
# Setup: make the eegad package importable (also when it was not installed with `pip install -e .`)
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore", category=RuntimeWarning)   # MNE notices about head size and 'boundary' events
warnings.filterwarnings("ignore", category=FutureWarning)

_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "analysis_plan.json").exists()), None)
if _root is None:
    raise FileNotFoundError("Open this notebook from inside the repository (eeg-ad-demographics/notebooks).")
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

In [ ]:
import hashlib
import json

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

from eegad.config import load_context
from eegad.data import load_table
from eegad.reporting import fmt, fmt_ci, fmt_metrics, fmt_p, read_json, write_json

from eegad.design import (comparable_pair_concordance, covariate_adjusted_auc, eligible_education, eligible_sex_age,
                          match_1to1, matched_auc_ci, overlap_weighted_auc_ci)

ctx = load_context("04")
log, out, plan = ctx.log, ctx.out, ctx.plan
NB, SEED = ctx.n_boot, ctx.seed
R2 = plan["rq2"]
CAL = R2["brainlat_matching"]["caliper_years"]
AGE_CAL = R2["greek_comparable_pairs"]["age_caliper_years"]
loso = read_json(out / "results" / "loso.json")


def with_score(subjects, name):
    pred = load_table(out / "predictions" / f"oof_{name}_RF_11.csv")
    return subjects.merge(pred[["subject_id", "p"]].rename(columns={"p": "p_eeg"}), on="subject_id", how="left")


subj_bl = with_score(load_table(out / "derived" / "brainlat_subjects.csv"), "BrainLat")
subj_gr = with_score(load_table(out / "derived" / "greek_subjects.csv"), "Greek")
design = read_json(out / "tables" / "confounding_design.json")
f_pairs = out / "derived" / "design_pairs_brainlat.csv"
assert hashlib.sha1(f_pairs.read_bytes()).hexdigest()[:12] == design["pairs_sha1"], "pairs differ from the design"
pairs_bl = load_table(f_pairs)
N_PAIRS = design["n_pairs_1to1"]
CONF = []


def conf_keys(cohort, name, auc, lo, hi):
    log.keys_from(f"rq2.{cohort}.{name}", {"value": auc, "lo": lo, "hi": hi})

## RQ2a - BrainLat

In [ ]:
P_BL = subj_bl.set_index("subject_id")["p_eeg"]
CONF.append({"cohort": "BrainLat", "analysis": "Full cohort", **dict(zip(("AUC", "lo", "hi"), loso["BrainLat"]["AUC"])),
             "n": f"{len(subj_bl)} participants"})
conf_keys("brainlat", "full", *loso["BrainLat"]["AUC"])
enough_pairs = N_PAIRS >= R2["brainlat_matching"]["min_pairs"]
if N_PAIRS >= 2:
    a_m = matched_auc_ci(pairs_bl, P_BL, NB, SEED)
    aucs_r = []
    for k in range(ctx.rt["n_tiebreaks"]):
        pr = match_1to1(subj_bl, CAL, rng=np.random.default_rng(SEED + 1 + k))
        if len(pr) >= 2:
            aucs_r.append(roc_auc_score(np.r_[np.ones(len(pr)), np.zeros(len(pr))],
                                        P_BL.loc[list(pr.ad_id) + list(pr.hc_id)].to_numpy()))
    print(f"Matched 1:1 (±{CAL} y): AUC {fmt_ci(*a_m)} | {N_PAIRS} pairs"
          + ("" if enough_pairs else " | BELOW the plan's minimum: descriptive only"))
    if aucs_r:
        print(f"  random tie-breaks ({len(aucs_r)}): median {np.median(aucs_r):.3f}, "
              f"range {np.min(aucs_r):.3f}–{np.max(aucs_r):.3f}")
        log.keys_from("rq2.brainlat.matched", {"tiebreak_median": float(np.median(aucs_r)),
                                               "tiebreak_min": float(np.min(aucs_r)),
                                               "tiebreak_max": float(np.max(aucs_r))})
    CONF.append({"cohort": "BrainLat", "analysis": f"Education-matched 1:1 (±{CAL} y)",
                 **dict(zip(("AUC", "lo", "hi"), a_m)), "n": f"{N_PAIRS} pairs"})
    conf_keys("brainlat", "matched", *a_m)
    log.key("rq2.brainlat.matched.n_pairs", N_PAIRS)
    log.record("RQ2a", f"BrainLat: AUC in 1:1 education-matched pairs (±{CAL} y)",
               f"{fmt_ci(*a_m)}; {N_PAIRS} pairs" + (f"; tie-breaks: median {np.median(aucs_r):.3f}, range "
                                                     f"{np.min(aucs_r):.3f}–{np.max(aucs_r):.3f}" if aucs_r else ""),
               "Results; Table 2", "" if enough_pairs else "below the minimum number of pairs: descriptive only")
else:
    a_m = (np.nan, np.nan, np.nan)
    log.alert("Fewer than 2 matched pairs: the matched AUC cannot be computed.")

c_bl = comparable_pair_concordance(subj_bl.dropna(subset=["years_education", "p_eeg"]),
                                   eligible_education(R2["brainlat_comparable_pairs"]["education_caliper_years"]),
                                   "p_eeg", NB, SEED)
print(f"Concordance in {c_bl['n_pairs']} comparable pairs (education ±2 y): {fmt_ci(c_bl['C'], c_bl['lo'], c_bl['hi'])}")
CONF.append({"cohort": "BrainLat", "analysis": "Comparable-pair concordance (education ±2 y)",
             "AUC": c_bl["C"], "lo": c_bl["lo"], "hi": c_bl["hi"], "n": f"{c_bl['n_pairs']} pairs"})
conf_keys("brainlat", "comparable_pairs", c_bl["C"], c_bl["lo"], c_bl["hi"])
log.key("rq2.brainlat.comparable_pairs.n_pairs", c_bl["n_pairs"])
log.record("RQ2a", "BrainLat: concordance in comparable pairs (education ±2 y)",
           f"{fmt_ci(c_bl['C'], c_bl['lo'], c_bl['hi'])}; {c_bl['n_pairs']} pairs", "Results; Table 2")

ow_bl = overlap_weighted_auc_ci(subj_bl, R2["brainlat_overlap"], "p_eeg", NB, SEED)
print(f"Overlap weights (age, sex, education): AUC {fmt_ci(ow_bl['AUC'], ow_bl['lo'], ow_bl['hi'])} | "
      f"effective size AD {ow_bl['ess_AD']:.1f}, HC {ow_bl['ess_HC']:.1f}")
CONF.append({"cohort": "BrainLat", "analysis": "Overlap-weighted (age, sex, education)",
             "AUC": ow_bl["AUC"], "lo": ow_bl["lo"], "hi": ow_bl["hi"], "n": f"ESS {ow_bl['ess_AD']:.0f}/{ow_bl['ess_HC']:.0f}"})
conf_keys("brainlat", "overlap", ow_bl["AUC"], ow_bl["lo"], ow_bl["hi"])
log.record("RQ2a", "BrainLat: AUC with overlap weights (age, sex, education)",
           f"{fmt_ci(ow_bl['AUC'], ow_bl['lo'], ow_bl['hi'])}; ESS AD {ow_bl['ess_AD']:.1f}, HC {ow_bl['ess_HC']:.1f}",
           "Results; Table 2")
aa_bl = covariate_adjusted_auc(subj_bl, ctx.cov_brainlat, "p_eeg", NB, SEED)
print(f"Covariate-adjusted AUC (Janes & Pepe; age, sex, education): {fmt_ci(aa_bl['AUC'], aa_bl['lo'], aa_bl['hi'])} "
      f"(n = {aa_bl['n']}; check)")
CONF.append({"cohort": "BrainLat", "analysis": "Covariate-adjusted AUC (age, sex, education)",
             "AUC": aa_bl["AUC"], "lo": aa_bl["lo"], "hi": aa_bl["hi"], "n": f"{aa_bl['n']} participants"})
conf_keys("brainlat", "adjusted", aa_bl["AUC"], aa_bl["lo"], aa_bl["hi"])
log.record("RQ2a", "BrainLat: covariate-adjusted AUC (Janes & Pepe; age, sex, education)",
           fmt_ci(aa_bl["AUC"], aa_bl["lo"], aa_bl["hi"]), "Results; Table 2", "check; outside the reading rule")
survives = [enough_pairs and np.isfinite(a_m[1]) and a_m[1] > 0.5, ow_bl["lo"] > 0.5]
v = log.verdict("RQ2a", "survives" if all(survives) else ("survives in part" if any(survives) else "not demonstrated"))
print(f"Reading (plan): {v} (1:1 pairs: {survives[0]}; overlap: {survives[1]})")

## RQ2b - Greek cohort

In [ ]:
CONF.append({"cohort": "Greek", "analysis": "Full cohort", **dict(zip(("AUC", "lo", "hi"), loso["Greek"]["AUC"])),
             "n": f"{len(subj_gr)} participants"})
conf_keys("greek", "full", *loso["Greek"]["AUC"])
ow_gr = overlap_weighted_auc_ci(subj_gr, R2["greek_overlap"], "p_eeg", NB, SEED)
print(f"Overlap weights (age, sex): AUC {fmt_ci(ow_gr['AUC'], ow_gr['lo'], ow_gr['hi'])} | "
      f"effective size AD {ow_gr['ess_AD']:.1f}, HC {ow_gr['ess_HC']:.1f}")
CONF.append({"cohort": "Greek", "analysis": "Overlap-weighted (age, sex)", "AUC": ow_gr["AUC"], "lo": ow_gr["lo"],
             "hi": ow_gr["hi"], "n": f"ESS {ow_gr['ess_AD']:.0f}/{ow_gr['ess_HC']:.0f}"})
conf_keys("greek", "overlap", ow_gr["AUC"], ow_gr["lo"], ow_gr["hi"])
c_gr = comparable_pair_concordance(subj_gr.dropna(subset=["Age", "sex_fem", "p_eeg"]), eligible_sex_age(AGE_CAL),
                                   "p_eeg", NB, SEED)
print(f"Concordance in {c_gr['n_pairs']} pairs of the same sex with age ±{AGE_CAL} y: {fmt_ci(c_gr['C'], c_gr['lo'], c_gr['hi'])}")
CONF.append({"cohort": "Greek", "analysis": f"Comparable-pair concordance (same sex, age ±{AGE_CAL} y)",
             "AUC": c_gr["C"], "lo": c_gr["lo"], "hi": c_gr["hi"], "n": f"{c_gr['n_pairs']} pairs"})
conf_keys("greek", "comparable_pairs", c_gr["C"], c_gr["lo"], c_gr["hi"])
log.key("rq2.greek.comparable_pairs.n_pairs", c_gr["n_pairs"])
aa_gr = covariate_adjusted_auc(subj_gr, ctx.cov_greek, "p_eeg", NB, SEED)
print(f"Covariate-adjusted AUC (Janes & Pepe; age, sex): {fmt_ci(aa_gr['AUC'], aa_gr['lo'], aa_gr['hi'])} "
      f"(n = {aa_gr['n']}; check)")
CONF.append({"cohort": "Greek", "analysis": "Covariate-adjusted AUC (age, sex)", "AUC": aa_gr["AUC"], "lo": aa_gr["lo"],
             "hi": aa_gr["hi"], "n": f"{aa_gr['n']} participants"})
conf_keys("greek", "adjusted", aa_gr["AUC"], aa_gr["lo"], aa_gr["hi"])
log.record("RQ2b", "Greek: covariate-adjusted AUC (Janes & Pepe; age, sex)", fmt_ci(aa_gr["AUC"], aa_gr["lo"], aa_gr["hi"]),
           "Results; Table 2", "check; outside the reading rule")
v = log.verdict("RQ2b", "survives" if ow_gr["lo"] > 0.5 else "not demonstrated")
print(f"Reading (plan): {v}")
log.record("RQ2b", "Greek: AUC with overlap weights (age, sex)",
           f"{fmt_ci(ow_gr['AUC'], ow_gr['lo'], ow_gr['hi'])}; ESS AD {ow_gr['ess_AD']:.1f}, HC {ow_gr['ess_HC']:.1f}",
           "Results; Table 2", f"reading: {v}")
log.record("RQ2b", f"Greek: concordance in pairs of the same sex and age ±{AGE_CAL} y",
           f"{fmt_ci(c_gr['C'], c_gr['lo'], c_gr['hi'])}; {c_gr['n_pairs']} pairs", "Results; Table 2")
TAB_CONF = pd.DataFrame(CONF)
display(TAB_CONF.round(3))
TAB_CONF.to_csv(out / "tables" / "T2b_confounding_by_design.csv", index=False, encoding="utf-8-sig")